# GeoProspectNet — single comprehensive notebook

**Everything in one place.** Data download → processing → EDA → smoke test → training → discovery → MWe → Zanskar validation → benchmarking → ablation → cross-basin → modality analysis → all figures + tables.

Designed to run end-to-end on a single Kaggle GPU session with `Internet: ON`. The auto-downloads in cell 3 pull every public dataset programmatically — you don't need to upload anything manually.

## Outputs

- `outputs/maps/prospectivity.tif` — continental p(geothermal) GeoTIFF (the headline)
- `outputs/results/discoveries_with_mwe.csv` — ranked candidate sites with MWe estimates
- `outputs/results/blind_validation.csv` — Zanskar Big Blind percentile check
- `outputs/results/table1_main_results.csv` — LOF-CV vs all baselines incl. Mordensky-2025 capture rates
- `outputs/results/table2_ablation.csv` — architectural ablation
- `outputs/results/cross_basin_transfer.json` — Basin & Range → Cascades transfer
- `outputs/figures/figure*.{png,pdf}` — 7 publication figures

## Sections

1. Setup & dependencies
2. Data download (auto, ~10 min)
3. Data processing (~30-60 min on CPU)
4. EDA / coverage report
5. Smoke test
6. Train discovery model (~1-2 hr GPU)
7. Continental discovery + MWe + Zanskar (HEADLINE)
8. Benchmarking (LOF-CV, ~5-10 hr GPU)
9. Ablation (~3 hr GPU)
10. Cross-basin transfer
11. Modality analysis
12. All figures + tables
13. Statistical tests
14. Archive outputs

## 1. Setup & dependencies

In [ ]:
!pip install -q rasterio rioxarray geopandas shapely pyproj fiona libpysal esda cartopy xlrd openpyxl elevation 2>&1 | tail -3

In [ ]:
import os, sys, subprocess, json
REPO = '/kaggle/working/Geothermal2' if os.path.exists('/kaggle') else os.getcwd()
if not os.path.isdir(REPO + '/src'):
    !git clone https://github.com/YOUR_GITHUB/Geothermal2.git $REPO || cp -R /kaggle/input/geoprospectnet-source/* $REPO/
os.chdir(REPO); sys.path.insert(0, REPO)
for d in ('data/raw/labels', 'data/raw/geophysics/smu_heatflow', 'data/raw/geochemistry',
         'data/raw/geology', 'data/raw/satellite', 'data/processed/splits',
         'data/metadata', 'outputs/checkpoints', 'outputs/results',
         'outputs/figures', 'outputs/maps'):
    os.makedirs(d, exist_ok=True)
print('working in', os.getcwd())

## 2. Data download — fully automated

Pulls every dataset from a working public URL. Total ~500 MB on disk after extraction; the giant intermediate ETOPO1 (322 MB) and SMU bulk (378 MB) are deleted after the relevant pieces are extracted.

In [ ]:
import urllib.request, zipfile, shutil

def fetch(url, dst, chunk=1<<16):
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        print(f'[skip] {dst}'); return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    print(f'  -> {dst}')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=300) as r, open(dst, 'wb') as f:
        while True:
            buf = r.read(chunk)
            if not buf: break
            f.write(buf)
    print(f'     {os.path.getsize(dst)/1e6:.1f} MB')

# 2a. Geophysics — USGS gravity GeoTIFFs
fetch('https://mrdata.usgs.gov/geophysics/gravity/USgrv_iso_SDD_geog.tif',
      'data/raw/geophysics/isostatic_gravity.tif')
fetch('https://mrdata.usgs.gov/geophysics/gravity/USgrv_cba_SDD_geog.tif',
      'data/raw/geophysics/bouguer_gravity.tif')

# 2b. USGS Quaternary faults
fetch('https://earthquake.usgs.gov/static/lfs/nshm/qfaults/Qfaults_GIS.zip',
      '/tmp/qfaults.zip')
if not os.path.exists('data/raw/geology/qfaults/SHP'):
    with zipfile.ZipFile('/tmp/qfaults.zip') as z:
        z.extractall('data/raw/geology/qfaults')

# 2c. USGS magnetic anomaly (Esri Grid -> we convert to GeoTIFF)
fetch('https://mrdata.usgs.gov/magnetic/USmag_origmrg.zip', '/tmp/usmag.zip')
if not os.path.exists('data/raw/geophysics/magnetic_anomaly.tif'):
    with zipfile.ZipFile('/tmp/usmag.zip') as z:
        z.extractall('/tmp/usmag')
    import rasterio
    with rasterio.open('/tmp/usmag/usmag_origmrg/w001001.adf') as src:
        prof = src.profile.copy(); prof.update(driver='GTiff', compress='lzw')
        with rasterio.open('data/raw/geophysics/magnetic_anomaly.tif', 'w', **prof) as dst:
            dst.write(src.read(1), 1)
    print('  converted Esri Grid -> GeoTIFF')

# 2d. SMU heat-flow bulk archive
fetch('https://gdr.openei.org/files/1704/Download%20Data%20Files.zip', '/tmp/smu_bulk.zip')
want = ['Download Data Files/Original Source Submissions/staging.smu_hf_view_materialized.zip',
        'Download Data Files/Original Source Submissions/staging.smu_bht_view_materialized.zip',
        'Download Data Files/Combined Sources into Each Data Type/core.template_heatflow_materialized.zip',
        'Download Data Files/Combined Sources into Each Data Type/core.surface_site_county_state_materialized_view.zip']
with zipfile.ZipFile('/tmp/smu_bulk.zip') as z:
    for w in want:
        z.extract(w, '/tmp/smu')
for inner in [w for w in want if 'staging' in w or 'heatflow' in w]:
    p = f'/tmp/smu/{inner}'
    with zipfile.ZipFile(p) as z: z.extractall('data/raw/geophysics/smu_heatflow/')
# Surface sites -> thermal-springs
ss_zip = '/tmp/smu/Download Data Files/Combined Sources into Each Data Type/core.surface_site_county_state_materialized_view.zip'
with zipfile.ZipFile(ss_zip) as z: z.extractall('/tmp/smu/sites')
import pandas as pd
ss = pd.read_csv('/tmp/smu/sites/core.surface_site_county_state_materialized_view.csv', low_memory=False)
ss = ss.dropna(subset=['latitude','longitude'])
ss = ss[(ss.latitude.between(31,49)) & (ss.longitude.between(-125,-103))]
ss.rename(columns={'latitude':'lat','longitude':'lon','max_temperature':'spring_T'}
)[['lat','lon','spring_T']].to_csv('data/raw/geochemistry/thermal_springs.csv', index=False)
print(f'  thermal_springs.csv: {len(ss)} western-US sites')

# 2e. USGS GEOTHERM (geochemistry)
fetch('https://gdr.openei.org/files/194/GEOTHERM_ALL.xls', 'data/raw/geochemistry/geotherm.xls')
if not os.path.exists('data/raw/geochemistry/geotherm.csv'):
    df = pd.read_excel('data/raw/geochemistry/geotherm.xls', engine='xlrd')
    df.columns = [c.lower().strip() for c in df.columns]
    df.to_csv('data/raw/geochemistry/geotherm.csv', index=False)
    print(f'  geotherm.csv: {len(df)} water-chemistry analyses')

# 2f. EIA-860 -> operating geothermal plants (labels)
fetch('https://www.eia.gov/electricity/data/eia860/archive/xls/eia8602023.zip', '/tmp/eia860.zip')
if not os.path.exists('data/raw/labels/eia_geothermal_plants.csv'):
    with zipfile.ZipFile('/tmp/eia860.zip') as z:
        z.extract('2___Plant_Y2023.xlsx', '/tmp')
        z.extract('3_1_Generator_Y2023.xlsx', '/tmp')
    plants = pd.read_excel('/tmp/2___Plant_Y2023.xlsx', header=1)
    gens = pd.read_excel('/tmp/3_1_Generator_Y2023.xlsx', header=1, sheet_name='Operable')
    geo = gens[gens['Energy Source 1'].astype(str).str.upper() == 'GEO']
    out = plants[plants['Plant Code'].isin(geo['Plant Code'].unique())] \
        [['Plant Code','Plant Name','State','Latitude','Longitude']] \
        .rename(columns={'Plant Code':'code','Plant Name':'name','State':'state',
                         'Latitude':'lat','Longitude':'lon'}) \
        .dropna(subset=['lat','lon'])
    out.to_csv('data/raw/labels/eia_geothermal_plants.csv', index=False)
    out.to_csv('data/raw/labels/usgs_identified_systems.csv', index=False)
    print(f'  EIA: {len(out)} operating geothermal plants')
    os.makedirs('data/raw/labels/operating_plants', exist_ok=True)
    os.makedirs('data/raw/labels/developing_plants', exist_ok=True)

# 2g. Smithsonian Holocene volcanoes (WFS CSV)
fetch('https://webservices.volcano.si.edu/geoserver/GVP-VOTW/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=GVP-VOTW:Smithsonian_VOTW_Holocene_Volcanoes&outputFormat=csv',
      'data/raw/geology/holocene_volcanoes.csv')

# 2h. ETOPO1 -> western-US elevation (clip to bbox)
fetch('https://www.ngdc.noaa.gov/mgg/global/relief/ETOPO1/data/ice_surface/grid_registered/georeferenced_tiff/ETOPO1_Ice_g_geotiff.zip',
      '/tmp/etopo1.zip')
if not os.path.exists('data/raw/geology/srtm_elevation.tif'):
    with zipfile.ZipFile('/tmp/etopo1.zip') as z:
        z.extractall('/tmp/etopo1')
    import rasterio
    from rasterio.windows import from_bounds
    with rasterio.open('/tmp/etopo1/ETOPO1_Ice_g_geotiff.tif') as src:
        win = from_bounds(-125, 31, -103, 49, src.transform)
        data = src.read(1, window=win)
        prof = src.profile.copy()
        prof.update({'height': data.shape[0], 'width': data.shape[1],
                     'transform': rasterio.windows.transform(win, src.transform),
                     'compress': 'lzw'})
        with rasterio.open('data/raw/geology/srtm_elevation.tif', 'w', **prof) as dst:
            dst.write(data, 1)
    print(f'  elevation: {data.shape}, {int(data.min())}-{int(data.max())} m')
    os.remove('/tmp/etopo1/ETOPO1_Ice_g_geotiff.tif')

# Cleanup giant intermediates
for p in ['/tmp/smu_bulk.zip', '/tmp/etopo1.zip', '/tmp/usmag.zip', '/tmp/eia860.zip']:
    if os.path.exists(p): os.remove(p)
for p in ['/tmp/smu', '/tmp/etopo1', '/tmp/usmag']:
    if os.path.isdir(p): shutil.rmtree(p, ignore_errors=True)

print('\n=== DATA INVENTORY ===')
!find data/raw -type f -size +1k | xargs ls -lh 2>/dev/null | awk '{print $5, $9}'

## 3. Data processing — build the 247K-cell western-US grid + all 4 modality arrays

In [ ]:
!python -m src.data.process_geophysics 2>&1 | tail -15

In [ ]:
!python -m src.data.process_geochemistry 2>&1 | tail -10

In [ ]:
!python -m src.data.process_thermal 2>&1 | tail -5     # skipped if no Landsat tif

In [ ]:
!python -m src.data.process_geology 2>&1 | tail -10

In [ ]:
!python -m src.data.build_labels 2>&1 | tail -15
!python -m src.data.build_splits 2>&1 | tail -5
!python -m src.data.build_neighbors 2>&1 | tail -3

## 4. Exploratory data analysis

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path

P = Path('data/processed')
grid = pd.read_csv(P / 'grid_coordinates.csv')
labels = np.load(P / 'labels.npy')
train_mask = np.load(P / 'train_mask.npy')
masks = np.load(P / 'modality_masks.npy')
fields = pd.read_csv('data/metadata/known_fields_details.csv')
provinces = pd.read_csv('data/metadata/tectonic_provinces.csv')

print(f'Grid:      {len(grid):,} cells')
print(f'Positives (labeled): {int(labels[train_mask].sum())}')
print(f'Negatives (labeled): {int(train_mask.sum() - labels[train_mask].sum())}')
print(f'Class ratio (labeled): 1 : {(train_mask.sum() - labels[train_mask].sum()) / max(1, labels[train_mask].sum()):.1f}')
print(f'Unique fields: {len(fields)}')
print()
print('Modality coverage:')
for i, m in enumerate(['geophysics','geochemistry','thermal','geology']):
    print(f'  {m:14s} {masks[:, i].mean():.3f}')
print()
print('Positive cells per tectonic province:')
pos_grid = provinces.iloc[np.flatnonzero(labels == 1)]
print(pos_grid['province'].value_counts().to_string())
print()
print('Known fields per state:')
print(fields['name'].apply(lambda s: 'unknown').value_counts().head() if 'state' not in fields else fields['state'].value_counts().head().to_string())

In [ ]:
# Visualise positives + coverage
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax = axes[0]
n_rows = int(grid['row'].max())+1; n_cols = int(grid['col'].max())+1
img = np.zeros((n_rows, n_cols)); img[grid['row'].values, grid['col'].values] = masks.sum(axis=1)
ax.imshow(img[::-1,:], extent=[-125,-103,31,49], cmap='viridis', vmin=0, vmax=4)
ax.scatter(fields['lon'], fields['lat'], c='red', s=20, marker='^', label=f'{len(fields)} known fields', edgecolor='white', linewidth=0.5)
ax.set_title('Modality coverage + known fields'); ax.legend()

ax = axes[1]
pos_idx = np.flatnonzero(labels == 1)
ax.scatter(grid.iloc[pos_idx]['lon'], grid.iloc[pos_idx]['lat'], c='red', s=4, label=f'{len(pos_idx)} positive cells', alpha=0.6)
neg_sample = np.random.choice(np.flatnonzero(train_mask & (labels==0)), size=min(2000, int((train_mask & (labels==0)).sum())), replace=False)
ax.scatter(grid.iloc[neg_sample]['lon'], grid.iloc[neg_sample]['lat'], c='gray', s=2, label='negatives (sampled)', alpha=0.3)
ax.set_xlim(-125,-103); ax.set_ylim(31,49); ax.legend(); ax.set_title('Labels')
plt.tight_layout(); plt.show()

## 5. CPU smoke test

In [ ]:
!python -m src.training.train --config configs/cpu_smoke.yaml --fold 0 --seed 42 2>&1 | tail -10

## 6. Train discovery model (GPU)

In [ ]:
!python -m src.training.train --config configs/default.yaml --random --seed 42 2>&1 | tail -20

## 7. HEADLINE — Continental discovery + MWe + Zanskar validation

In [ ]:
!python -m src.evaluation.discovery --config configs/default.yaml 2>&1 | tail -10
!python -m src.evaluation.mwe_estimation --config configs/default.yaml 2>&1 | tail -20
!python -m src.evaluation.blind_validation --config configs/default.yaml 2>&1 | tail -10

In [ ]:
import pandas as pd
if Path('outputs/results/discoveries_with_mwe.csv').exists():
    df = pd.read_csv('outputs/results/discoveries_with_mwe.csv')
    print(f'\n{len(df)} candidate sites, cumulative {df.mwe_central.sum():.0f} MWe '
          f'[{df.mwe_low.sum():.0f}-{df.mwe_high.sum():.0f}]')
    display(df.head(15))
if Path('outputs/results/blind_validation.csv').exists():
    print('\n=== Zanskar blind validation ===')
    display(pd.read_csv('outputs/results/blind_validation.csv'))

In [ ]:
# Headline figure
!python -m src.visualization.prospectivity_map 2>&1 | tail -3
from IPython.display import Image, display
if Path('outputs/figures/figure4_prospectivity_map.png').exists():
    display(Image('outputs/figures/figure4_prospectivity_map.png'))

## 8. Benchmarking (LOF-CV vs all baselines, with Mordensky-2025 capture rates)

Long. Skip the `--all_folds` flag if Kaggle GPU hours are tight.

In [ ]:
# Quick: 5 folds (~30-40 min)
!python -m src.training.train --config configs/default.yaml --quick 2>&1 | tail -10
!python -m src.training.train_baselines --config configs/default.yaml --quick --baseline all 2>&1 | tail -20
# Full LOF-CV (uncomment for the paper-grade run, ~10h):
# !python -m src.training.train --config configs/default.yaml --all_folds 2>&1 | tail -10
# !python -m src.training.train_baselines --config configs/default.yaml --all_folds --baseline all 2>&1 | tail -10
!python -m src.evaluation.cross_validation 2>&1 | tail -20

In [ ]:
if Path('outputs/results/table1_main_results.csv').exists():
    display(pd.read_csv('outputs/results/table1_main_results.csv'))

## 9. Ablation

In [ ]:
!python -m src.evaluation.ablation --config configs/default.yaml --ablation configs/ablation.yaml --n_folds 5 2>&1 | tail -15
if Path('outputs/results/table2_ablation.csv').exists():
    display(pd.read_csv('outputs/results/table2_ablation.csv'))

## 10. Cross-basin transfer

In [ ]:
!python -m src.evaluation.cross_basin --config configs/default.yaml \
    --train_provinces Basin_and_Range \
    --test_provinces Cascades Snake_River_Plain 2>&1 | tail -25

## 11. Modality + spatial analysis

In [ ]:
!python -m src.evaluation.modality_analysis --config configs/default.yaml --n_repeats 5 2>&1 | tail -15
!python -m src.evaluation.spatial_analysis --config configs/default.yaml --n_folds 3 2>&1 | tail -10
!python -m src.evaluation.embedding_viz --config configs/default.yaml --n_sample 5000 2>&1 | tail -5

## 12. All figures + LaTeX tables

In [ ]:
!bash scripts/08_figures.sh 2>&1 | tail -15
for fn in sorted(os.listdir('outputs/figures')):
    if fn.endswith('.png'):
        print(fn); display(Image(f'outputs/figures/{fn}'))

## 13. Statistical tests

In [ ]:
!python -m src.evaluation.statistical_tests 2>&1 | tail -20

## 14. Archive everything

In [ ]:
WORKING = '/kaggle/working' if os.path.exists('/kaggle') else 'outputs'
!tar czf {WORKING}/geoprospect_outputs.tar.gz outputs/results outputs/checkpoints outputs/figures outputs/maps 2>/dev/null
!ls -lh {WORKING}/geoprospect_outputs.tar.gz